# Structured Extraction Fine-Tuning

**Goal:** fine-tune a small language model to extract structured records from unstructured text, with strict schema adherence and layered evaluation.

---

## Why Fine-Tune for Extraction?

Prompt engineering works for simple extraction, but production pipelines need:

| Requirement | Prompt-only risk | Fine-tuning benefit |
|---|---|---|
| Consistent JSON schema across thousands of inputs | LLMs omit fields, hallucinate keys, or change casing | Weights learn the exact schema shape |
| Low latency per document | Long system prompts eat tokens and time | Short prompts, same quality |
| Reliable handling of edge cases (missing data, ambiguous fields) | Each edge case needs a new example in the prompt | The model learns edge-case conventions from training data |
| Stable behavior across model updates | Provider model changes can silently break schema | Your adapter locks in the behavior you tested |

## What This Notebook Covers

1. Define a target extraction schema with validation rules
2. Build a supervised dataset of document → structured-record pairs
3. Format examples for chat-style SFT
4. Split train/validation with domain-aware stratification
5. Configure LoRA fine-tuning
6. **Evaluate with a layered framework**: schema validity, field accuracy, edge-case handling, and regression detection
7. Decide when extraction fine-tuning is worth the maintenance cost

## Pipeline Diagram

```
Raw documents (invoices, support tickets, bios, etc.)
        │
        ▼
┌──────────────────────┐
│   Fine-tuned model   │  Extract structured JSON
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│   Schema validator   │  Pydantic / JSON Schema check
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│   Field-level eval   │  Precision, recall, exact match per field
└──────────┬───────────┘
           │
           ▼
   Validated structured records
```

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q pandas numpy scikit-learn datasets transformers peft trl accelerate pydantic

In [ ]:
import json
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from datasets import Dataset
from pydantic import BaseModel, ValidationError, field_validator
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RUN_TRAINING = False
RUN_GENERATION_EVAL = False
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = ARTIFACT_DIR / "extraction-lora"

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifacts:    {ARTIFACT_DIR}")
print(f"Base model:   {BASE_MODEL}")
print(f"Training:     {RUN_TRAINING}")
print(f"Gen eval:     {RUN_GENERATION_EVAL}")

## 2. Define the Target Schema

Before building any dataset, define the **exact schema** the model must produce. This is the contract between the model and downstream systems.

### Schema Design Rules

- every field has a type, a description, and allowed values or format
- optional fields use `null` rather than omitting the key (structural consistency matters more than saving a few tokens)
- enums are closed lists, not free text
- dates use ISO 8601
- monetary amounts are objects with `amount` and `currency`

### Why Pydantic?

We use a Pydantic model as the single source of truth. It serves three purposes:

1. **Documentation** — the schema is self-describing
2. **Validation** — every model output is validated before use
3. **Eval** — schema violations are the first eval layer

In [ ]:
class MoneyAmount(BaseModel):
    amount: float
    currency: str  # ISO 4217 (USD, EUR, GBP, etc.)


class InvoiceRecord(BaseModel):
    vendor_name: str
    invoice_number: str
    invoice_date: str  # ISO 8601: YYYY-MM-DD
    due_date: Optional[str] = None  # ISO 8601 or null
    line_items: list[dict]  # [{"description": ..., "quantity": ..., "unit_price": ...}]
    subtotal: MoneyAmount
    tax: Optional[MoneyAmount] = None
    total: MoneyAmount
    payment_terms: Optional[str] = None
    notes: Optional[str] = None

    @field_validator("invoice_date", "due_date", mode="before")
    @classmethod
    def validate_date(cls, v):
        if v is None:
            return v
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", v):
            raise ValueError(f"Date must be YYYY-MM-DD, got: {v}")
        return v


# Verify the schema works
sample = InvoiceRecord(
    vendor_name="Acme Corp",
    invoice_number="INV-2025-0042",
    invoice_date="2025-11-15",
    due_date="2025-12-15",
    line_items=[
        {"description": "Widget A", "quantity": 10, "unit_price": 25.00},
        {"description": "Widget B", "quantity": 5, "unit_price": 40.00},
    ],
    subtotal=MoneyAmount(amount=450.00, currency="USD"),
    tax=MoneyAmount(amount=36.00, currency="USD"),
    total=MoneyAmount(amount=486.00, currency="USD"),
    payment_terms="Net 30",
    notes=None,
)

print("Schema validation passed:")
print(json.dumps(sample.model_dump(), indent=2))

In [ ]:
# Show the JSON Schema that the model must conform to
schema = InvoiceRecord.model_json_schema()
print(json.dumps(schema, indent=2))

## 3. Build a Supervised Dataset

Each training example pairs an **unstructured invoice document** with the **target structured JSON**.

### Dataset Design Principles

For extraction fine-tuning, the dataset must cover:

- **format variety** — different invoice layouts and wording styles
- **edge cases** — missing fields, ambiguous dates, multi-currency, zero-tax invoices
- **schema discipline** — every target is schema-valid before training

### Real-World Sources

In production, your datasets come from:

- OCR output from scanned documents paired with human-verified structured records
- legacy database exports matched to their source documents
- manually annotated samples from your document pipeline

In [ ]:
documents = [
    {
        "doc_id": "DOC-001",
        "doc_type": "standard",
        "text": (
            "INVOICE\n"
            "Bright Solutions LLC\n"
            "Invoice #: BS-2025-1184\n"
            "Date: November 3, 2025\n"
            "Due: December 3, 2025\n\n"
            "Description              Qty    Unit Price\n"
            "Cloud hosting (monthly)    1     $2,400.00\n"
            "SSL certificate renewal    3       $149.99\n\n"
            "Subtotal: $2,849.97\n"
            "Tax (8%): $227.99\n"
            "Total Due: $3,077.96\n\n"
            "Payment Terms: Net 30\n"
            "Please remit payment by the due date."
        ),
        "target": {
            "vendor_name": "Bright Solutions LLC",
            "invoice_number": "BS-2025-1184",
            "invoice_date": "2025-11-03",
            "due_date": "2025-12-03",
            "line_items": [
                {"description": "Cloud hosting (monthly)", "quantity": 1, "unit_price": 2400.00},
                {"description": "SSL certificate renewal", "quantity": 3, "unit_price": 149.99},
            ],
            "subtotal": {"amount": 2849.97, "currency": "USD"},
            "tax": {"amount": 227.99, "currency": "USD"},
            "total": {"amount": 3077.96, "currency": "USD"},
            "payment_terms": "Net 30",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-002",
        "doc_type": "minimal",
        "text": (
            "From: DataForge Inc.\n"
            "Inv 44210 | Jan 15 2026\n"
            "API calls (Jan): 1,200,000 @ $0.002 = $2,400.00\n"
            "Amount due: $2,400.00"
        ),
        "target": {
            "vendor_name": "DataForge Inc.",
            "invoice_number": "44210",
            "invoice_date": "2026-01-15",
            "due_date": None,
            "line_items": [
                {"description": "API calls (Jan)", "quantity": 1200000, "unit_price": 0.002},
            ],
            "subtotal": {"amount": 2400.00, "currency": "USD"},
            "tax": None,
            "total": {"amount": 2400.00, "currency": "USD"},
            "payment_terms": None,
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-003",
        "doc_type": "international",
        "text": (
            "FACTURE\n"
            "Europa Design SARL\n"
            "Facture No: ED-9921\n"
            "Date de facture: 22/02/2026\n"
            "Echeance: 22/03/2026\n\n"
            "Prestation                     Qte    Prix HT\n"
            "Consultation UX (jours)          5    450,00 EUR\n"
            "Rapport livrable                 1    800,00 EUR\n\n"
            "Sous-total HT: 3 050,00 EUR\n"
            "TVA (20%): 610,00 EUR\n"
            "Total TTC: 3 660,00 EUR\n\n"
            "Conditions: 30 jours nets"
        ),
        "target": {
            "vendor_name": "Europa Design SARL",
            "invoice_number": "ED-9921",
            "invoice_date": "2026-02-22",
            "due_date": "2026-03-22",
            "line_items": [
                {"description": "Consultation UX (jours)", "quantity": 5, "unit_price": 450.00},
                {"description": "Rapport livrable", "quantity": 1, "unit_price": 800.00},
            ],
            "subtotal": {"amount": 3050.00, "currency": "EUR"},
            "tax": {"amount": 610.00, "currency": "EUR"},
            "total": {"amount": 3660.00, "currency": "EUR"},
            "payment_terms": "30 jours nets",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-004",
        "doc_type": "handwritten_style",
        "text": (
            "Invoice from: Quick Print Co\n"
            "inv no 7788\n"
            "date: march 5th, 2026\n\n"
            "- 500 business cards  $75\n"
            "- 200 flyers (A5)     $120\n"
            "total $195\n"
            "no tax (exempt)\n"
            "pay within 14 days please"
        ),
        "target": {
            "vendor_name": "Quick Print Co",
            "invoice_number": "7788",
            "invoice_date": "2026-03-05",
            "due_date": None,
            "line_items": [
                {"description": "500 business cards", "quantity": 500, "unit_price": 0.15},
                {"description": "200 flyers (A5)", "quantity": 200, "unit_price": 0.60},
            ],
            "subtotal": {"amount": 195.00, "currency": "USD"},
            "tax": None,
            "total": {"amount": 195.00, "currency": "USD"},
            "payment_terms": "Net 14",
            "notes": "Tax exempt",
        },
    },
    {
        "doc_id": "DOC-005",
        "doc_type": "multi_item",
        "text": (
            "INVOICE\n"
            "TechSupply Partners\n"
            "Invoice Number: TSP-2026-0335\n"
            "Invoice Date: 2026-04-01\n"
            "Payment Due: 2026-05-01\n\n"
            "Items:\n"
            "1. Laptop stand (ergonomic)   x10   $89.99 each\n"
            "2. USB-C hub                  x10   $34.50 each\n"
            "3. Wireless mouse             x10   $24.00 each\n"
            "4. Monitor arm (dual)         x5    $179.00 each\n\n"
            "Subtotal: $2,379.90\n"
            "Sales Tax (7.25%): $172.52\n"
            "Shipping: included\n"
            "Total: $2,552.42\n\n"
            "Terms: Net 30. Contact billing@techsupply.example for questions."
        ),
        "target": {
            "vendor_name": "TechSupply Partners",
            "invoice_number": "TSP-2026-0335",
            "invoice_date": "2026-04-01",
            "due_date": "2026-05-01",
            "line_items": [
                {"description": "Laptop stand (ergonomic)", "quantity": 10, "unit_price": 89.99},
                {"description": "USB-C hub", "quantity": 10, "unit_price": 34.50},
                {"description": "Wireless mouse", "quantity": 10, "unit_price": 24.00},
                {"description": "Monitor arm (dual)", "quantity": 5, "unit_price": 179.00},
            ],
            "subtotal": {"amount": 2379.90, "currency": "USD"},
            "tax": {"amount": 172.52, "currency": "USD"},
            "total": {"amount": 2552.42, "currency": "USD"},
            "payment_terms": "Net 30",
            "notes": "Shipping included",
        },
    },
    {
        "doc_id": "DOC-006",
        "doc_type": "credit_note",
        "text": (
            "CREDIT NOTE\n"
            "GreenLeaf Supplies\n"
            "Credit Note #: CN-4412\n"
            "Original Invoice: GLS-2025-8890\n"
            "Date: 2026-01-20\n\n"
            "Returned item:\n"
            "Defective paper shredder   -1   ($129.00)\n\n"
            "Total Credit: -$129.00\n"
            "This credit will be applied to your next invoice."
        ),
        "target": {
            "vendor_name": "GreenLeaf Supplies",
            "invoice_number": "CN-4412",
            "invoice_date": "2026-01-20",
            "due_date": None,
            "line_items": [
                {"description": "Defective paper shredder (return)", "quantity": -1, "unit_price": 129.00},
            ],
            "subtotal": {"amount": -129.00, "currency": "USD"},
            "tax": None,
            "total": {"amount": -129.00, "currency": "USD"},
            "payment_terms": None,
            "notes": "Credit note for GLS-2025-8890. Applied to next invoice.",
        },
    },
    {
        "doc_id": "DOC-007",
        "doc_type": "gbp_invoice",
        "text": (
            "Camden Consulting Ltd\n"
            "VAT Reg: GB 123 4567 89\n"
            "Invoice: CC-2258\n"
            "Date: 14 March 2026\n"
            "Due: 14 April 2026\n\n"
            "Strategy workshop (2 days)     1   GBP 3,200.00\n"
            "Travel expenses                1   GBP   420.50\n\n"
            "Net: GBP 3,620.50\n"
            "VAT @ 20%: GBP 724.10\n"
            "Total: GBP 4,344.60\n\n"
            "Payment: Bank transfer within 30 days"
        ),
        "target": {
            "vendor_name": "Camden Consulting Ltd",
            "invoice_number": "CC-2258",
            "invoice_date": "2026-03-14",
            "due_date": "2026-04-14",
            "line_items": [
                {"description": "Strategy workshop (2 days)", "quantity": 1, "unit_price": 3200.00},
                {"description": "Travel expenses", "quantity": 1, "unit_price": 420.50},
            ],
            "subtotal": {"amount": 3620.50, "currency": "GBP"},
            "tax": {"amount": 724.10, "currency": "GBP"},
            "total": {"amount": 4344.60, "currency": "GBP"},
            "payment_terms": "Net 30",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-008",
        "doc_type": "ambiguous_date",
        "text": (
            "StellarOps Inc. / Invoice 9034\n"
            "Issued: 04/07/2026\n\n"
            "Managed Kubernetes cluster (monthly)  1  $1,850.00\n"
            "Premium support add-on                1    $300.00\n\n"
            "Total: $2,150.00\n"
            "Due upon receipt"
        ),
        "target": {
            "vendor_name": "StellarOps Inc.",
            "invoice_number": "9034",
            "invoice_date": "2026-04-07",
            "due_date": None,
            "line_items": [
                {"description": "Managed Kubernetes cluster (monthly)", "quantity": 1, "unit_price": 1850.00},
                {"description": "Premium support add-on", "quantity": 1, "unit_price": 300.00},
            ],
            "subtotal": {"amount": 2150.00, "currency": "USD"},
            "tax": None,
            "total": {"amount": 2150.00, "currency": "USD"},
            "payment_terms": "Due upon receipt",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-009",
        "doc_type": "verbose_email",
        "text": (
            "Hi Sandra,\n\n"
            "Please find attached the invoice for this month. Here are the details:\n\n"
            "From: Pixel Perfect Design Studio\n"
            "Invoice number: PPD-0617\n"
            "Date: February 28, 2026\n\n"
            "We completed the following work:\n"
            "- Homepage redesign (40 hours at $125/hr): $5,000\n"
            "- Icon pack creation (custom, 24 icons): $960\n\n"
            "The subtotal is $5,960.00. No tax applies as design services are "
            "exempt in our jurisdiction.\n\n"
            "Total: $5,960.00\n\n"
            "Our standard terms are Net 45. Please let me know if you have any "
            "questions.\n\n"
            "Best regards,\n"
            "Jamie"
        ),
        "target": {
            "vendor_name": "Pixel Perfect Design Studio",
            "invoice_number": "PPD-0617",
            "invoice_date": "2026-02-28",
            "due_date": None,
            "line_items": [
                {"description": "Homepage redesign", "quantity": 40, "unit_price": 125.00},
                {"description": "Icon pack creation (custom, 24 icons)", "quantity": 1, "unit_price": 960.00},
            ],
            "subtotal": {"amount": 5960.00, "currency": "USD"},
            "tax": None,
            "total": {"amount": 5960.00, "currency": "USD"},
            "payment_terms": "Net 45",
            "notes": "Design services tax exempt",
        },
    },
    {
        "doc_id": "DOC-010",
        "doc_type": "single_item",
        "text": (
            "INVOICE\n"
            "Zenith Security Corp\n"
            "INV: ZSC-1001\n"
            "Date: 2026-03-31\n"
            "Due: 2026-04-30\n\n"
            "Annual security audit   1   $18,500.00\n\n"
            "Subtotal: $18,500.00\n"
            "Tax: $0.00\n"
            "Total: $18,500.00\n\n"
            "Terms: Net 30"
        ),
        "target": {
            "vendor_name": "Zenith Security Corp",
            "invoice_number": "ZSC-1001",
            "invoice_date": "2026-03-31",
            "due_date": "2026-04-30",
            "line_items": [
                {"description": "Annual security audit", "quantity": 1, "unit_price": 18500.00},
            ],
            "subtotal": {"amount": 18500.00, "currency": "USD"},
            "tax": {"amount": 0.00, "currency": "USD"},
            "total": {"amount": 18500.00, "currency": "USD"},
            "payment_terms": "Net 30",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-011",
        "doc_type": "noisy_ocr",
        "text": (
            "INIVOICE\n"
            "RapidShip Logistics\n"
            "lnv #: RL-7765\n"
            "Dat: 2O26-O2-10\n"
            "Due. 2026-03-10\n\n"
            "Freight (pallets x4)     4   $680.00 ea\n"
            "Customs brokerage        1   $250.00\n\n"
            "Subtota1: $2,970.00\n"
            "Tax: $0.O0\n"
            "Tota1: $2,970.00\n\n"
            "Net 3O"
        ),
        "target": {
            "vendor_name": "RapidShip Logistics",
            "invoice_number": "RL-7765",
            "invoice_date": "2026-02-10",
            "due_date": "2026-03-10",
            "line_items": [
                {"description": "Freight (pallets x4)", "quantity": 4, "unit_price": 680.00},
                {"description": "Customs brokerage", "quantity": 1, "unit_price": 250.00},
            ],
            "subtotal": {"amount": 2970.00, "currency": "USD"},
            "tax": {"amount": 0.00, "currency": "USD"},
            "total": {"amount": 2970.00, "currency": "USD"},
            "payment_terms": "Net 30",
            "notes": None,
        },
    },
    {
        "doc_id": "DOC-012",
        "doc_type": "jpy_invoice",
        "text": (
            "請求書\n"
            "Sakura Tech Co., Ltd.\n"
            "請求書番号: ST-20260401\n"
            "発行日: 2026年4月1日\n"
            "支払期限: 2026年4月30日\n\n"
            "ソフトウエア開発 (3月)    1   ¥1,200,000\n"
            "サーバー運用費             1   ¥180,000\n\n"
            "小計: ¥1,380,000\n"
            "消費税 (10%): ¥138,000\n"
            "合計: ¥1,518,000\n\n"
            "お支払い条件: 月末締め翌月末払い"
        ),
        "target": {
            "vendor_name": "Sakura Tech Co., Ltd.",
            "invoice_number": "ST-20260401",
            "invoice_date": "2026-04-01",
            "due_date": "2026-04-30",
            "line_items": [
                {"description": "Software development (March)", "quantity": 1, "unit_price": 1200000},
                {"description": "Server operation fee", "quantity": 1, "unit_price": 180000},
            ],
            "subtotal": {"amount": 1380000, "currency": "JPY"},
            "tax": {"amount": 138000, "currency": "JPY"},
            "total": {"amount": 1518000, "currency": "JPY"},
            "payment_terms": "End of month, payment by end of following month",
            "notes": None,
        },
    },
]

df = pd.DataFrame(documents)
print(f"Dataset: {len(df)} documents")
print(df[["doc_id", "doc_type"]].to_string(index=False))

## 4. Validate Every Target Against the Schema

**Non-negotiable rule:** every training target must pass schema validation before it enters the dataset. If training targets are schema-invalid, the model learns to produce schema-invalid outputs.

This is the most common mistake in extraction fine-tuning: teams skip target validation and wonder why the model produces malformed JSON in production.

In [ ]:
validation_results = []
for _, row in df.iterrows():
    try:
        record = InvoiceRecord(**row["target"])
        validation_results.append({"doc_id": row["doc_id"], "valid": True, "error": None})
    except ValidationError as e:
        validation_results.append({"doc_id": row["doc_id"], "valid": False, "error": str(e)})

val_df = pd.DataFrame(validation_results)
passed = val_df["valid"].sum()
failed = len(val_df) - passed
print(f"Schema validation: {passed} passed, {failed} failed")
if failed > 0:
    print("\nFailed records:")
    print(val_df[~val_df["valid"]][["doc_id", "error"]])
else:
    print("All training targets are schema-valid.")

## 5. Format Examples for SFT

Each training example is a chat turn:

- **system**: extraction instructions + JSON schema
- **user**: the raw document text
- **assistant**: the target JSON

### System Prompt Design for Extraction

The system prompt should include the full JSON schema definition. This is different from style tuning where the system prompt is short. For extraction, the schema IS the instruction.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = '''You are a structured data extraction engine.
Given an invoice document (text, OCR output, or email), extract the information into a JSON object that strictly conforms to this schema:

{
  "vendor_name": "string (required)",
  "invoice_number": "string (required)",
  "invoice_date": "string, YYYY-MM-DD (required)",
  "due_date": "string, YYYY-MM-DD or null",
  "line_items": [{"description": "string", "quantity": number, "unit_price": number}],
  "subtotal": {"amount": number, "currency": "ISO 4217 string"},
  "tax": {"amount": number, "currency": "ISO 4217 string"} or null,
  "total": {"amount": number, "currency": "ISO 4217 string"},
  "payment_terms": "string or null",
  "notes": "string or null"
}

Rules:
- Output ONLY valid JSON. No markdown fences, no explanation, no commentary.
- All dates must be YYYY-MM-DD regardless of the input date format.
- Use null for missing optional fields. Do not omit keys.
- Infer currency from symbols or text ($ = USD, EUR/euro = EUR, GBP/pound = GBP, yen = JPY).
- If a field is ambiguous, make your best interpretation and preserve the original wording in notes.
- Negative amounts are valid (credit notes, returns).
- Handle OCR errors gracefully (e.g., "O" for "0", "1" for "l").'''


def to_chat_record(row):
    target_json = json.dumps(row["target"], indent=2, ensure_ascii=False)
    return {
        "doc_id": row["doc_id"],
        "doc_type": row["doc_type"],
        "messages": [
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": target_json},
        ],
    }


chat_records = [to_chat_record(row) for _, row in df.iterrows()]
chat_df = pd.DataFrame(chat_records)

# Show one example
print("Example formatted record:")
print(f"  System prompt: {len(EXTRACTION_SYSTEM_PROMPT)} chars")
print(f"  User (document): {len(chat_records[0]['messages'][1]['content'])} chars")
print(f"  Assistant (target JSON): {len(chat_records[0]['messages'][2]['content'])} chars")

In [ ]:
# Save JSONL for reuse
jsonl_path = ARTIFACT_DIR / "extraction_sft.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for rec in chat_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Saved: {jsonl_path}")

## 6. Train / Validation Split

### Split Strategy for Extraction

Random splitting is acceptable only when documents are independent. Watch for:

| Leakage Risk | Example | Fix |
|---|---|---|
| Same vendor in both sets | Model memorizes vendor-specific formatting | Split by vendor |
| Same template in both sets | Model memorizes the layout, not the extraction skill | Split by template/doc_type |
| Time leak | Newer invoices reference older ones | Split by date |

For this demo (small, synthetic, independent docs), we stratify by `doc_type` to keep variety in both sets.

In [ ]:
train_df, val_df = train_test_split(
    chat_df,
    test_size=0.25,
    random_state=SEED,
    stratify=chat_df["doc_type"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"\nTrain doc types: {dict(Counter(train_df['doc_type']))}")
print(f"Val doc types:   {dict(Counter(val_df['doc_type']))}")

## 7. Prepare Chat Text for Training

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_text(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(lambda row: {"text": render_text(row["messages"])})
val_dataset = val_dataset.map(lambda row: {"text": render_text(row["messages"])})

# Token length analysis
train_lengths = [len(tokenizer(ex["text"]).input_ids) for ex in train_dataset]
val_lengths = [len(tokenizer(ex["text"]).input_ids) for ex in val_dataset]

length_report = pd.DataFrame({
    "split": ["train", "validation"],
    "count": [len(train_lengths), len(val_lengths)],
    "mean_tokens": [np.mean(train_lengths), np.mean(val_lengths)],
    "max_tokens": [max(train_lengths), max(val_lengths)],
    "p95_tokens": [np.percentile(train_lengths, 95), np.percentile(val_lengths, 95)],
})
length_report

## 8. Fine-Tuning with LoRA

### Why LoRA for Extraction?

- Schema adherence is a narrow behavioral objective — LoRA adapters learn it efficiently
- Small adapters are cheaper to version, deploy, and roll back than full model copies
- You can swap adapters per document domain (invoices, receipts, contracts) without reloading the base model

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

MAX_SEQ_LENGTH = int(max(max(train_lengths), max(val_lengths)) + 64)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)

print(f"LoRA rank: {peft_config.r}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")
print(f"Epochs: {training_args.num_train_epochs}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=(
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    ),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Trainer ready")

In [ ]:
if RUN_TRAINING:
    result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(result)
    print(f"Adapter saved to: {OUTPUT_DIR}")
else:
    print("Training skipped. Set RUN_TRAINING = True to run.")

## 9. Evaluation Framework

Extraction evaluation is more structured than style evaluation. You can define **objective, automated metrics** for most of what matters.

### The Four Eval Layers

| Layer | What It Measures | Automated? | When to Use |
|---|---|---|---|
| **1. Schema Validity** | Can the output be parsed as JSON and does it pass Pydantic validation? | Yes | Every output, every time |
| **2. Field Accuracy** | Does each extracted field match the reference? | Yes | Every validation example |
| **3. Edge-Case Coverage** | Does the model handle missing fields, OCR noise, non-USD currency, negative amounts? | Yes (with tagged test slices) | Regression testing |
| **4. Failure Mode Analysis** | What specific patterns cause failures? | Semi-auto | After each training run |

### Why Schema Validity Comes First

If the output is not valid JSON, nothing else matters. Schema validity is a **gating metric**: if it drops below 95%, fix that before evaluating field accuracy.

### Field-Level Metrics

For each field, compute:

- **exact match** — the extracted value equals the reference exactly
- **normalized match** — after lowercasing, stripping whitespace, normalizing dates
- **numeric tolerance** — amounts within a small epsilon (for floating-point formatting differences)

In [ ]:
def evaluate_schema_validity(output_text):
    result = {"parseable_json": False, "schema_valid": False, "errors": []}
    try:
        parsed = json.loads(output_text)
        result["parseable_json"] = True
    except json.JSONDecodeError as e:
        result["errors"].append(f"JSON parse error: {e}")
        return result

    try:
        InvoiceRecord(**parsed)
        result["schema_valid"] = True
    except ValidationError as e:
        result["errors"].append(f"Schema validation: {e}")
    return result


# Demo: test with a valid output
valid_output = json.dumps(documents[0]["target"], indent=2)
print("Valid output test:")
print(evaluate_schema_validity(valid_output))

# Demo: test with broken JSON
broken_output = '{"vendor_name": "Acme",  "invoice_date": "Nov 3"}'  # missing required fields
print("\nBroken output test:")
print(evaluate_schema_validity(broken_output))

In [ ]:
def evaluate_field_accuracy(predicted, reference, numeric_tol=0.01):
    results = {}

    # Simple string fields
    for field in ["vendor_name", "invoice_number", "invoice_date", "due_date", "payment_terms"]:
        pred_val = predicted.get(field)
        ref_val = reference.get(field)
        if ref_val is None and pred_val is None:
            results[field] = {"match": True, "type": "null_match"}
        elif ref_val is None or pred_val is None:
            results[field] = {"match": False, "type": "null_mismatch", "pred": pred_val, "ref": ref_val}
        elif str(pred_val).strip().lower() == str(ref_val).strip().lower():
            results[field] = {"match": True, "type": "exact"}
        else:
            results[field] = {"match": False, "type": "mismatch", "pred": pred_val, "ref": ref_val}

    # Money fields
    for field in ["subtotal", "tax", "total"]:
        pred_val = predicted.get(field)
        ref_val = reference.get(field)
        if ref_val is None and pred_val is None:
            results[field] = {"match": True, "type": "null_match"}
        elif ref_val is None or pred_val is None:
            results[field] = {"match": False, "type": "null_mismatch"}
        else:
            amount_ok = abs(pred_val.get("amount", 0) - ref_val.get("amount", 0)) <= numeric_tol
            currency_ok = pred_val.get("currency", "").upper() == ref_val.get("currency", "").upper()
            results[field] = {"match": amount_ok and currency_ok, "type": "numeric"}

    # Line items count
    pred_items = predicted.get("line_items", [])
    ref_items = reference.get("line_items", [])
    results["line_item_count"] = {
        "match": len(pred_items) == len(ref_items),
        "type": "count",
        "pred": len(pred_items),
        "ref": len(ref_items),
    }

    return results


# Demo: compare a perfect extraction
pred_json = documents[0]["target"]
ref_json = documents[0]["target"]
field_results = evaluate_field_accuracy(pred_json, ref_json)
for field, result in field_results.items():
    status = "PASS" if result["match"] else "FAIL"
    print(f"  {status}  {field}")

In [ ]:
def run_full_eval(outputs_and_refs):
    schema_stats = {"total": 0, "json_ok": 0, "schema_ok": 0}
    field_stats = {}
    per_doc = []

    for item in outputs_and_refs:
        schema_stats["total"] += 1
        sv = evaluate_schema_validity(item["output"])

        doc_result = {"doc_id": item["doc_id"], "json_ok": sv["parseable_json"], "schema_ok": sv["schema_valid"]}

        if sv["parseable_json"]:
            schema_stats["json_ok"] += 1
        if sv["schema_valid"]:
            schema_stats["schema_ok"] += 1
            parsed = json.loads(item["output"])
            fa = evaluate_field_accuracy(parsed, item["reference"])
            for field, result in fa.items():
                if field not in field_stats:
                    field_stats[field] = {"correct": 0, "total": 0}
                field_stats[field]["total"] += 1
                if result["match"]:
                    field_stats[field]["correct"] += 1
                doc_result[f"field_{field}"] = result["match"]

        per_doc.append(doc_result)

    # Summary
    n = schema_stats["total"]
    print("EXTRACTION EVAL SUMMARY")
    print("=" * 55)
    print(f"  JSON parseable:    {schema_stats['json_ok']}/{n}  ({schema_stats['json_ok']/n:.0%})")
    print(f"  Schema valid:      {schema_stats['schema_ok']}/{n}  ({schema_stats['schema_ok']/n:.0%})")
    print()
    print(f"  {'Field':<22} {'Accuracy':>10}")
    print(f"  {'-'*22} {'-'*10}")
    for field, stats in sorted(field_stats.items()):
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        print(f"  {field:<22} {acc:>9.0%}")

    return pd.DataFrame(per_doc)


# Demo: simulate a perfect eval run using training targets as both output and reference
demo_eval_data = [
    {"doc_id": row["doc_id"], "output": json.dumps(row["target"]), "reference": row["target"]}
    for _, row in df.iterrows()
]

eval_report = run_full_eval(demo_eval_data)

## 10. Edge-Case Test Slices

Schema validity and field accuracy averaged across all examples can hide category-specific failures. Define **test slices** to catch them.

### What Extraction Slices to Track

| Slice | Why | What Breaks |
|---|---|---|
| Missing optional fields | Model may hallucinate a value instead of null | `due_date`, `tax`, `notes` |
| Non-USD currency | Model may default to USD | EUR, GBP, JPY invoices |
| Negative amounts | Credit notes, returns | `total`, `subtotal` sign |
| OCR noise | Typos in field names or numbers | All fields |
| Ambiguous dates | MM/DD vs DD/MM | `invoice_date`, `due_date` |
| Verbose / email-wrapped | Extraction from conversational text | All fields |
| Multi-item invoices | Correct line item count | `line_items` |

In [ ]:
EVAL_SLICES = {
    "missing_optional": ["DOC-002", "DOC-004", "DOC-008"],
    "non_usd": ["DOC-003", "DOC-007", "DOC-012"],
    "negative_amounts": ["DOC-006"],
    "ocr_noise": ["DOC-011"],
    "ambiguous_date": ["DOC-008"],
    "verbose_email": ["DOC-009"],
    "multi_item": ["DOC-005"],
}

print("EVAL SLICES")
print("=" * 50)
for slice_name, doc_ids in EVAL_SLICES.items():
    print(f"  {slice_name:<22} {len(doc_ids)} docs: {', '.join(doc_ids)}")

# With the eval framework already built, you would run:
# slice_results = {}
# for slice_name, doc_ids in EVAL_SLICES.items():
#     slice_data = [d for d in eval_data if d["doc_id"] in doc_ids]
#     slice_results[slice_name] = run_full_eval(slice_data)

## 11. Generation Eval: Base vs. Fine-Tuned

After training, compare the base model and the fine-tuned model on the validation set.

The comparison measures:

1. schema validity rate
2. per-field accuracy
3. per-slice accuracy
4. specific failure patterns

In [ ]:
from transformers import pipeline as hf_pipeline
from peft import PeftModel


def generate_extraction(generator, document_text, max_new_tokens=600):
    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
        {"role": "user", "content": document_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=None)
    generated = output[0]["generated_text"]
    return generated[len(prompt):].strip()


if RUN_GENERATION_EVAL:
    # Load base model
    base_gen = hf_pipeline(
        "text-generation",
        model=AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto"),
        tokenizer=tokenizer,
    )

    # Load fine-tuned model
    tuned_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto")
    tuned_model = PeftModel.from_pretrained(tuned_base, str(OUTPUT_DIR))
    tuned_gen = hf_pipeline("text-generation", model=tuned_model, tokenizer=tokenizer)

    # Build eval data from val set
    doc_lookup = {row["doc_id"]: row for _, row in df.iterrows()}
    base_eval_data = []
    tuned_eval_data = []

    for _, row in val_df.iterrows():
        doc = doc_lookup[row["doc_id"]]
        base_output = generate_extraction(base_gen, doc["text"])
        tuned_output = generate_extraction(tuned_gen, doc["text"])
        base_eval_data.append({"doc_id": row["doc_id"], "output": base_output, "reference": doc["target"]})
        tuned_eval_data.append({"doc_id": row["doc_id"], "output": tuned_output, "reference": doc["target"]})

    print("BASE MODEL EVAL")
    base_report = run_full_eval(base_eval_data)

    print("\nFINE-TUNED MODEL EVAL")
    tuned_report = run_full_eval(tuned_eval_data)
else:
    print("Generation eval skipped. Set RUN_GENERATION_EVAL = True after training.")

## 12. Regression Detection

Every time you retrain or update the dataset, run the eval suite and compare to the previous best.

### What Constitutes a Regression?

| Metric | Regression Threshold | Action |
|---|---|---|
| Schema validity drops | Any drop below 95% | Block deployment |
| Field accuracy drops on any field | > 5% drop | Investigate before deploying |
| New failure on a previously-passing slice | Any new failure | Root-cause before deploying |
| Line item count accuracy drops | Any drop | Check if new training data changed item parsing conventions |

### Storing Eval Baselines

Save the eval results after each successful training run. The next run compares against the saved baseline.

In [ ]:
def save_eval_baseline(eval_results, run_name):
    baseline_path = ARTIFACT_DIR / f"eval_baseline_{run_name}.json"
    baseline = {
        "run_name": run_name,
        "timestamp": datetime.now().isoformat(),
        "results": eval_results.to_dict(orient="records"),
    }
    baseline_path.write_text(json.dumps(baseline, indent=2, default=str), encoding="utf-8")
    print(f"Baseline saved: {baseline_path}")
    return baseline_path


def check_regression(current_results, baseline_path, schema_threshold=0.95, field_drop_threshold=0.05):
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    prev = pd.DataFrame(baseline["results"])

    issues = []

    # Schema validity regression
    curr_schema_rate = current_results["schema_ok"].mean()
    prev_schema_rate = prev["schema_ok"].mean()
    if curr_schema_rate < schema_threshold:
        issues.append(f"Schema validity {curr_schema_rate:.0%} below threshold {schema_threshold:.0%}")
    if curr_schema_rate < prev_schema_rate - field_drop_threshold:
        issues.append(f"Schema validity dropped: {prev_schema_rate:.0%} -> {curr_schema_rate:.0%}")

    # Field accuracy regressions
    field_cols = [c for c in current_results.columns if c.startswith("field_")]
    for col in field_cols:
        if col in prev.columns:
            curr_acc = current_results[col].mean()
            prev_acc = prev[col].mean()
            if curr_acc < prev_acc - field_drop_threshold:
                issues.append(f"{col}: {prev_acc:.0%} -> {curr_acc:.0%} (dropped)")

    if issues:
        print("REGRESSION DETECTED")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("No regressions detected.")

    return issues


# Save demo baseline
demo_baseline_path = save_eval_baseline(eval_report, "demo_v1")

## 13. When Fine-Tuning for Extraction Is Worth It

### Decision Framework

| Question | Yes Signal | No Signal |
|---|---|---|
| Does the schema change rarely? | Stable for months | Weekly schema changes |
| Do you have 200+ labeled document-JSON pairs? | Enough annotated data | Fewer than 50 examples |
| Does the prompt-only approach drop below 90% schema validity? | Prompt baseline is unreliable | Prompt already works well |
| Are you processing 1000+ documents/day? | Scale justifies investment | Low volume, manual QC is fine |
| Can you afford to maintain a training pipeline? | CI/CD for model retraining exists | No MLOps capacity |
| Is latency a constraint? | Need fast inference without long prompts | Latency is not critical |

### Cost of Getting It Wrong

Fine-tuning extraction specifically has a **high cost of silent failure**. Unlike style tuning where a slightly off tone is annoying but harmless, a wrong invoice amount or missing tax field can cause real financial or legal errors.

This means:

- always validate outputs with Pydantic or JSON Schema in production
- never trust the model alone for financial data
- treat fine-tuning as improving the **hit rate**, not as removing the need for validation
- monitor schema validity rate and field accuracy in production dashboards

### Alternatives to Weigh

| Alternative | When to Prefer It |
|---|---|
| Prompt engineering + structured output mode | Schema is simple, volume is low, model supports JSON mode |
| Few-shot prompting with retrieval | You have many schema variants and cannot train one adapter per schema |
| Rule-based extraction (regex, templates) | Documents follow a very consistent layout |
| Hybrid: rules for structured fields + LLM for free-text fields | Mix of structured and unstructured content |

## 14. Key Takeaways

### What We Built

- A **schema-first extraction pipeline** where Pydantic defines the contract before any training begins
- A **supervised dataset** with format variety, edge cases, and pre-validated targets
- A **LoRA fine-tuning setup** for a small instruct model
- A **four-layer eval framework**: schema validity → field accuracy → edge-case slices → regression detection

### Lessons for Extraction Fine-Tuning

1. **Schema validation is a gating metric.** If the model cannot produce valid JSON, field accuracy is meaningless.

2. **Validate training targets.** If your training data has schema errors, so will your model.

3. **Test edge cases explicitly.** Average accuracy hides failures on OCR noise, non-USD currency, missing fields, and negative amounts.

4. **Build regression detection from day one.** Every retrain should compare against the previous baseline.

5. **Fine-tuning improves hit rate, not correctness guarantee.** Always validate outputs with Pydantic in production.

6. **Keep the schema in the system prompt and in the validator.** Two sources of truth that must stay synchronized.

### Production Checklist

- [ ] Schema defined in Pydantic and used for both training validation and inference validation
- [ ] Training targets 100% schema-valid
- [ ] Eval slices cover every known edge case
- [ ] Regression detection runs on every retrain
- [ ] Schema validity monitored in production
- [ ] Field accuracy tracked per field, not just averaged
- [ ] Model outputs never trusted without validation for financial or legal data